# 01 · Gene basis and inputs for the joint iNMF

**Object.** `thyAgeing_Fb_scell_multiome_nuclei_anno_2026-08-20.zarr` — 85,289 thymic fibroblasts
from dissociated scRNA-seq (the *cells* arm, 70,258) and multiome nuclei (the *nuclei* arm,
15,031), integrated with scVI. `X` is log1p(CP10K); `layers['counts']` holds the raw counts.

**What this notebook decides.**

1. The 173 low-complexity `FB_interlo_2` nuclei (median 646 genes) are dropped.
2. Whether the mature-adipose programme is captured in nuclei at all — per-gene detection rates,
   not a composite score, because a composite cannot separate "all cells slightly positive" from
   "some cells strongly positive". The answer (the transcribed core PLIN1 / AQP7 / FABP4 / GPD1 /
   PPP1R1A is shared, the terminal lipid-droplet markers ADIPOQ / CIDEA / LIPE / PLIN4 are
   cells-only) is what lets a shared adipose factor be accepted on the core genes alone.
3. The **gene basis** of the factorisation: genes highly variable in *both* arms. Highly variable
   genes are ranked per arm (`seurat_v3` on counts, batched by age group), the two rankings are
   intersected at a depth chosen so that the marker genes of the target programmes survive
   (`N_TOP = 6000`, giving 3,641 shared genes), technical families are removed, and every gene of
   the curated marker panels present in the object is force-included (30 further genes). Genes variable
   in only one arm are what split the earlier joint runs into modality-specific factors, which is
   why the intersection rather than the union is used.

**Outputs** (`data/nmf_joint/final/`): `basis_shared.txt` (the 3,671 genes), `inmf_input_counts.h5ad`
(raw counts, all genes; the fit restricts to the basis), `fb_light.h5ad` (metadata + UMAP + scVI
latent, no expression matrix — what every later notebook loads). The basis is checked against the
one the model was actually fitted on.

In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')
import hdf5plugin                      # registers the compression filters; must precede any h5 read
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

sys.path.insert(0, '/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final')
import fb_common as C

C.set_style()
RES, PLT = C.out_dirs('01_gene_basis')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
print(f'anndata {ad.__version__} | scanpy {sc.__version__}')
print(f'tables -> {RES}\nfigures -> {PLT}\ndata -> {C.FINAL_DATA}')

anndata 0.12.19 | scanpy 1.11.5
tables -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/final/01_gene_basis
figures -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/01_gene_basis
data -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final


## 1 · Load and audit

In [2]:
t0 = time.time()
adata = ad.read_h5ad(C.OBJ)            # HDF5 despite the .zarr extension
print(adata.shape, f'| loaded in {time.time() - t0:.0f}s')

# float64 on disk -> float32 halves the footprint before anything else touches it
if adata.X.dtype != np.float32:
    adata.X = adata.X.astype(np.float32)
adata.layers['counts'] = adata.layers['counts'].astype(np.float32)

# arm and the per-arm annotation. `taa_l4` is cells-only, `celltype_v3_L3` nuclei-only, and they
# share no labels -- that complementarity is why programmes (factors) are used rather than clusters.
adata.obs['arm'] = adata.obs['type'].astype(str).map({'cells': 'cells', 'nuclei': 'nuclei'})
assert adata.obs['arm'].notna().all(), 'unexpected value in obs["type"]'
lab = adata.obs['taa_l4'].astype(str)
is_nuc = (adata.obs['arm'] == 'nuclei').values
lab[is_nuc] = adata.obs['celltype_v3_L3'].astype(str)[is_nuc]
adata.obs['label'] = pd.Categorical(lab.replace({'nan': np.nan, 'None': np.nan}))

print(adata.obs['arm'].value_counts().to_string(), '\n')
print(pd.crosstab(adata.obs['label'], adata.obs['arm']).to_string())

(85289, 35530) | loaded in 6s


arm
cells     70258
nuclei    15031 

arm                cells  nuclei
label                           
FB_TGFB1               0    2603
FB_adipo               0    4178
FB_interlo             0    3545
FB_interlo_2           0     173
FB_interm              0    2116
FB_med                 0      28
FB_perilo              0    1078
FB_perilo_COLEC11      0    1310
Fb-adipo            5220       0
Fb-interlo          8595       0
Fb-interlo_PI16    10816       0
Fb-interm          15036       0
Fb-med              7523       0
Fb-perilo          12624       0
Fb-perilo_COLEC11  10444       0


In [3]:
# GUARD: X must be log1p(CP10K) and counts integral. Verified on a few rows rather than trusted.
ok = True
for i in [0, 1000, 50000, adata.n_obs - 1]:
    x = np.expm1(adata.X[i].toarray().ravel())
    cnt = adata.layers['counts'][i].toarray().ravel()
    nz = cnt > 0
    scale = np.median(x[nz] / cnt[nz] * cnt.sum())
    print(f'row {i:>6}: library {cnt.sum():>9.0f}  implied scale factor {scale:.1f}')
    ok &= np.allclose(scale, 1e4, rtol=1e-3)
assert ok, 'X is NOT log1p(CP10K)'
assert np.allclose(adata.layers['counts'].data, np.round(adata.layers['counts'].data)), 'counts not integral'
print('OK: X = log1p(CP10K), counts are integral')

row      0: library    154702  implied scale factor 10000.0
row   1000: library     10508  implied scale factor 10000.0
row  50000: library      4186  implied scale factor 10000.0
row  85288: library       823  implied scale factor 10000.0


OK: X = log1p(CP10K), counts are integral


In [4]:
# Drop the low-complexity nuclei group (173 nuclei, median 646 genes, intronic fraction 0.17)
keep = adata.obs['label'].astype(str) != 'FB_interlo_2'
print(f'dropping FB_interlo_2: {int((~keep).sum())} nuclei')
adata = adata[keep.values].copy()
adata.obs['label'] = adata.obs['label'].cat.remove_unused_categories()
print(adata.shape)

dropping FB_interlo_2: 173 nuclei


(85116, 35530)


## 2 · Is the mature-adipose programme captured in nuclei?

Two read-outs: a composite `score_genes` score on the strict mature-adipose panel (comparable
across cells), and the per-gene detection rate (fraction of cells with ≥ 1 count) in the adipose
label versus the best other label of the same arm. The detection fold is the honest answer to
"is this transcript class captured at all in nuclei".

In [5]:
SIG = C.load_signatures()
MATURE = [g for g in SIG['target']['adipose_mature'] if g in adata.var_names]
sc.tl.score_genes(adata, MATURE, score_name='adipose_mature', ctrl_size=50, random_state=0)

CORE = [g for g in ['ADIPOQ', 'PLIN1', 'PLIN4', 'CIDEC', 'CIDEA', 'LIPE', 'AQP7', 'FABP4', 'GPD1', 'PPP1R1A'] if g in adata.var_names]
idx = [adata.var_names.get_loc(g) for g in CORE]
det = pd.DataFrame(np.asarray((adata.layers['counts'][:, idx] > 0).todense()), columns=CORE, index=adata.obs_names)
det['arm'] = adata.obs['arm'].values
det['label'] = adata.obs['label'].astype(str).values
tab = (det.groupby(['arm', 'label'], observed=True)[CORE].mean() * 100).round(2)
tab = tab.join(det.groupby(['arm', 'label'], observed=True).size().rename('n'))
tab = tab[tab['n'] >= 20]
tab.to_csv(f'{RES}/adipose_gene_detection_by_label.csv')
print('detection rate (% of cells with >= 1 count) of mature-adipocyte genes:')
print(tab.to_string())

rows = {}
for arm in C.ARMS:
    t = tab.loc[arm]
    adipo = [i for i in t.index if 'adipo' in i.lower()][0]
    rest = t.drop(index=adipo)[CORE].max()
    rows[arm] = (t.loc[adipo, CORE] / rest.replace(0, np.nan)).round(1)
fold = pd.DataFrame(rows)
fold.to_csv(f'{RES}/adipose_gene_detection_fold.csv')
print('\nfold enrichment of detection in the adipose label over the best other label, per arm:')
print(fold.to_string())

target     8 panels, 117 genes | id_perilobular, id_interlobular, id_medullary, inflammatory, adipose_mature, ECM_synthesis, Fib_TGFb, myofib_contractile
nuisance   3 panels,  32 genes | IEG_dissoc, proliferation, lymphoid
split      2 panels,  10 genes | adipose_core, adipose_terminal
spatial   15 panels, 117 genes | Adipo_progenitor, Adipo_preadipocyte, Adipo_mature, Adipo_lipidsynth, Lipid_droplet_storage, Lipolysis, FA_uptake_transport, Lipogenesis_denovo, Lipid_master_regulators, Fib_collagen_ECM, Fib_crosslink, Fib_remodel, Fib_EMT_TF, Fib_myofibroblast, Fibrosis_curated_10


detection rate (% of cells with >= 1 count) of mature-adipocyte genes:
                          ADIPOQ  PLIN1  PLIN4  CIDEC  CIDEA  LIPE   AQP7  FABP4   GPD1  PPP1R1A      n
arm    label                                                                                           
cells  Fb-adipo             0.61  10.86   6.38  31.72   0.38  9.31  15.94  30.63  20.86    35.21   5220
       Fb-interlo           0.01   0.72   0.29   9.18   0.02  2.48   0.81   6.15   0.80     2.76   8595
       Fb-interlo_PI16      0.06   0.16   0.03   2.52   0.04  3.03   0.06   0.77   0.48     0.51  10816
       Fb-interm            0.03   0.39   0.14   8.23   0.03  1.10   0.48   4.23   0.86     5.65  15036
       Fb-med               0.00   0.15   0.03   0.33   0.00  1.20   0.35   7.08   0.13     0.97   7523
       Fb-perilo            0.02   0.59   0.17   5.73   0.02  2.84   1.24   5.09   2.68     8.41  12624
       Fb-perilo_COLEC11    0.02   0.12   0.15   2.21   0.00  1.23   0.92   8.65   0.89     2.57 

In [6]:
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('double', 55), sharey=True, layout='constrained')
for ax, arm in zip(axes, C.ARMS):
    d = adata.obs[adata.obs['arm'] == arm]
    order = d.groupby('label', observed=True)['adipose_mature'].median().sort_values(ascending=False).index
    data = [d.loc[d['label'] == l, 'adipose_mature'].values for l in order]
    ax.boxplot(data, showfliers=False, widths=0.6)
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(list(order), rotation=90)
    ax.axhline(0, color='0.6', lw=0.5)
    ax.set_title(f'{C.ARM_NAMES[arm]} — mature-adipose score (score_genes)')
    ax.grid(False)
axes[0].set_ylabel('adipose_mature score')
C.save(fig, f'{PLT}/adipose_mature_score_by_label.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/01_gene_basis/adipose_mature_score_by_label.pdf   183 x 55 mm


## 3 · The gene basis

One `seurat_v3` fit per arm at `N_MAX = 8000` genes, keeping the **rank**, so any smaller depth is
a slice of the same ranking rather than a refit. Batches are the age groups with ≥ 500 cells in
the arm. Then:

- the depth sweep below shows how many genes are shared at each `N_TOP` and how many of the core
  marker genes of each target programme survive the intersection;
- `N_TOP = 6000` is the smallest depth at which every core marker survives;
- technical families (mitochondrial, ribosomal, metallothionein, MALAT1/NEAT1 and other
  modality-marking housekeeping genes) are removed;
- every gene of the curated marker panels present in the object is force-included, because the
  panels are the acceptance criterion of the factorisation and a factor cannot form around a
  marker that is not in the basis.

The panel set force-included here is the one that was current when the basis was fixed
(2026-08-23): the six target panels of that date (perilobular, interlobular, medullary,
inflammatory, mature adipose, fibrotic/myofibroblast), the three nuisance panels and the
`parenchymal` panel that was later retired. The signature CSV has since been revised (the
fibrotic panel split into ECM synthesis and myofibroblast/contractile, `Fib_TGFb` promoted,
`parenchymal` folded into perilobular); those revisions changed how factors are *named*, not the
gene basis they were fitted on, which is why the historical set is spelled out below.

**The basis of record is the one the model was fitted on** (`data/nmf_joint/basis_shared.txt`,
3,671 genes; copied to `data/nmf_joint/final/`). The recomputation below is a reproducibility
check against it: the forced panel genes reproduce exactly, the well-ranked genes reproduce to
the rank (the top 1,000 of the cells ranking are > 99 % identical) and the basis reproduces to
~97 %. The residual ~5 % are rarely detected genes (unannotated loci, antisense transcripts,
olfactory / testis / immune genes seen in a handful of fibroblasts) whose regularised variance
is tied with hundreds of others near the 6,000-gene cut-off: with the same library versions and
the same cells, floating-point summation order in the variance estimate is enough to reorder
those ties from run to run, so the exact set of boundary genes is not reproducible to the gene.
The check fails if a curated panel gene differs, if more than 5 % of the basis differs, or if the
differing genes are not rarely detected (median detection ≥ 10 % of that of the basis genes).

In [7]:
N_MAX, HVG_BATCH, MIN_BATCH = 8000, 'age_group', 500
ranks = {}
for arm in C.ARMS:
    m = (adata.obs['arm'] == arm).values
    n_per = adata.obs.loc[m, HVG_BATCH].astype(str).value_counts()
    big = [b for b in n_per.index[n_per >= MIN_BATCH] if b not in ('pooled', 'nan', 'None')]
    rows = np.flatnonzero(m & adata.obs[HVG_BATCH].astype(str).isin(big).values)
    fit = ad.AnnData(X=adata.layers['counts'][rows],
                     obs=pd.DataFrame({HVG_BATCH: pd.Categorical(adata.obs[HVG_BATCH].astype(str).values[rows])},
                                      index=adata.obs_names[rows]),
                     var=pd.DataFrame(index=adata.var_names))
    sc.pp.highly_variable_genes(fit, n_top_genes=N_MAX, flavor='seurat_v3',
                                batch_key=HVG_BATCH if len(big) >= 2 else None)
    ranks[arm] = fit.var['highly_variable_rank'].reindex(adata.var_names).values
    print(f'{arm}: ranked {int(np.isfinite(ranks[arm]).sum()):,} genes over {len(rows):,} cells (batches {big})')
    del fit
ranks = pd.DataFrame(ranks, index=adata.var_names)
ranks.to_csv(f'{RES}/hvg_ranks_per_arm.csv')
tech = np.array([C.is_technical(g) for g in adata.var_names])
print(f'technical genes excluded: {int(tech.sum())}')

cells: ranked 17,934 genes over 70,258 cells (batches ['adult', 'infant', 'aged', 'paed'])


nuclei: ranked 8,000 genes over 14,240 cells (batches ['adult'])
technical genes excluded: 220


In [8]:
# core markers that MUST survive the intersection, per target programme
CORE_MARKERS = {'id_perilobular': ['COLEC11', 'COL15A1', 'ALDH1A2', 'PTN'],
                'id_interlobular': ['PI16', 'MFAP5', 'CD55', 'PCOLCE2'],
                'id_medullary': ['CCL19', 'CCL21', 'CD74'],
                'inflammatory': ['PTX3', 'IL1R1', 'CD44', 'TNFRSF12A'],
                'adipose_mature': SIG['split']['adipose_core'],
                'fibrotic_myofib': ['ACTA2', 'TGFB1', 'COL1A1', 'POSTN']}
rows = []
for n in [2000, 3000, 4000, 5000, 6000, 8000]:
    sh = ((ranks['cells'] < n) & (ranks['nuclei'] < n)).values & ~tech
    genes = set(adata.var_names[sh])
    row = {'n_top': n, 'n_shared': int(sh.sum())}
    for panel, core in CORE_MARKERS.items():
        row[panel] = f'{sum(g in genes for g in core)}/{len(core)}'
    row['missing'] = ', '.join(g for core in CORE_MARKERS.values() for g in core if g in adata.var_names and g not in genes)
    rows.append(row)
sweep = pd.DataFrame(rows).set_index('n_top')
sweep.to_csv(f'{RES}/hvg_depth_sweep.csv')
print('shared-HVG size and core-marker retention per depth:')
print(sweep.to_string())

shared-HVG size and core-marker retention per depth:
       n_shared id_perilobular id_interlobular id_medullary inflammatory adipose_mature fibrotic_myofib                                                                   missing
n_top                                                                                                                                                                            
2000        712            2/4             2/4          2/3          2/4            3/5             3/4  COLEC11, PTN, CD55, PCOLCE2, CD74, CD44, TNFRSF12A, GPD1, PPP1R1A, TGFB1
3000       1290            3/4             3/4          3/3          3/4            4/5             3/4                               COLEC11, PCOLCE2, TNFRSF12A, PPP1R1A, TGFB1
4000       1989            3/4             4/4          3/3          4/4            4/5             3/4                                                   COLEC11, PPP1R1A, TGFB1
5000       2781            3/4             4/4          3

In [9]:
N_TOP = 6000
shared = ((ranks['cells'] < N_TOP) & (ranks['nuclei'] < N_TOP)).values & ~tech

# the marker panels as they stood when the basis was fixed (2026-08-23), see the note above
PANELS_AT_BASIS_TIME = {
    'id_perilobular':  ['COL15A1', 'COLEC11', 'ALDH1A2', 'PTN', 'HAND2', 'LAMC3', 'DPT', 'ANGPTL1', 'RBP5'],
    'id_interlobular': SIG['target']['id_interlobular'],
    'id_medullary':    SIG['target']['id_medullary'],
    'inflammatory':    SIG['target']['inflammatory'],
    'adipose_mature':  SIG['target']['adipose_mature'],
    'fibrotic_myofib': ['SNAI1', 'SNAI2', 'TGFB1', 'ACTA2', 'TAGLN', 'THY1', 'LMCD1', 'PRG4', 'ALDH1A3',
                        'COL1A1', 'COL3A1', 'COL5A1', 'POSTN', 'CTHRC1', 'TNC', 'LRRC15'],
    'parenchymal':     ['C7', 'APOD', 'FBLN5', 'MFAP4', 'LUM', 'FMO2', 'GPX3', 'CFD', 'ADH1B', 'CEBPD', 'IGFBP5', 'IGFBP3'],
    **SIG['nuisance'], **SIG['split'],
}
panel_genes = sorted({g for v in PANELS_AT_BASIS_TIME.values() for g in v if g in adata.var_names})
forced = np.isin(adata.var_names, panel_genes) & ~tech
basis = shared | forced

print(f'shared HVGs (N_TOP={N_TOP}) : {int(shared.sum()):,}')
print(f'panel genes forced in       : {int((forced & ~shared).sum())} added')
print(f'FINAL BASIS                 : {int(basis.sum()):,} genes')

basis_df = pd.DataFrame({'gene': adata.var_names[basis], 'shared_hvg': shared[basis], 'panel_forced': (forced & ~shared)[basis]})
basis_df.to_csv(f'{C.FINAL_DATA}/genes_basis_shared_recomputed.csv', index=False)
with open(f'{C.FINAL_DATA}/basis_shared_recomputed.txt', 'w') as fh:
    fh.write('\n'.join(adata.var_names[basis]) + '\n')
print(f'-> {C.FINAL_DATA}/basis_shared_recomputed.txt')

shared HVGs (N_TOP=6000) : 3,638
panel genes forced in       : 30 added
FINAL BASIS                 : 3,668 genes
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/basis_shared_recomputed.txt


In [10]:
import re, shutil

# 1. the basis of record goes forward
shutil.copy(f'{C.NMF_DIR}/basis_shared.txt', f'{C.FINAL_DATA}/basis_shared.txt')
shutil.copy(f'{C.NMF_DIR}/genes_basis_shared.csv', f'{C.FINAL_DATA}/genes_basis_shared.csv')
W_genes = pd.read_csv(f'{C.INMF_DIR}/W_{C.TAG}.csv', index_col=0, nrows=1).columns
rec, new = set(W_genes), set(adata.var_names[basis])
gset = rec
print(f'-> {C.FINAL_DATA}/basis_shared.txt  ({len(gset):,} genes, the basis the model was fitted on)')

# 2. the recomputation against it
rec_df = pd.read_csv(f'{C.NMF_DIR}/genes_basis_shared.csv').set_index('gene')
extra, lost = sorted(new - rec), sorted(rec - new)
locus = re.compile(r'^A[CLPF]\d+\.\d+$|^LINC\d+|^MIR\d|-AS\d|^Z\d+\.\d+$|^U\d+\.\d+$')
print(f'\nrecorded basis {len(rec):,} | recomputed {len(new):,} | shared {len(rec & new):,} ({len(rec & new) / len(rec):.2%} of the recorded)')
print(f'forced panel genes: recorded {int(rec_df["panel_forced"].sum())}, recomputed {int(basis_df["panel_forced"].sum())}, '
      f'identical: {set(rec_df.index[rec_df["panel_forced"]]) == set(basis_df.loc[basis_df["panel_forced"], "gene"])}')
print(f'differences: {len(extra)} only recomputed, {len(lost)} only recorded; '
      f'{sum(bool(locus.search(g)) for g in extra + lost)} of {len(extra) + len(lost)} are unannotated loci or antisense transcripts')
# the differing genes are rarely detected -- their regularised variances are near-tied and the tail of the ranking is unstable
frac = lambda genes: pd.Series(np.asarray((adata.layers['counts'][:, [adata.var_names.get_loc(g) for g in genes]] > 0).mean(axis=0)).ravel(), index=genes)
d_diff, d_kept = frac(extra + lost), frac(sorted(rec & new))
print(f'fraction of cells detecting the gene -- differing genes: median {d_diff.median():.4f}, max {d_diff.max():.4f}; '
      f'genes in both bases: median {d_kept.median():.4f}')
print('  most-detected differing genes:', ', '.join(f'{g} {v:.3f}' for g, v in d_diff.nlargest(6).items()))
assert len(rec & new) / len(rec) >= 0.95, 'the recomputed basis differs from the recorded one by more than 5%'
assert not (set(panel_genes) & set(extra + lost)), 'a curated panel gene differs between the recorded and the recomputed basis'
assert d_diff.median() < 0.1 * d_kept.median(), 'the differing genes are not rarely-detected loci'
print('OK: the recorded basis is reproduced up to rarely-detected boundary genes')

# 3. every non-technical gene of the CURRENT target panels present in the object is in that basis
rows = []
for p, v in SIG['target'].items():
    in_obj = [g for g in v if g in adata.var_names]
    target = [g for g in in_obj if not C.is_technical(g)]
    rows.append({'panel': p, 'in_object': len(in_obj), 'technical_excluded': len(in_obj) - len(target),
                 'target': len(target), 'in_basis': sum(g in gset for g in target),
                 'missing': ', '.join(g for g in target if g not in gset)})
chk = pd.DataFrame(rows).set_index('panel')
print('\ncoverage of the current target panels by the basis (Fib_TGFb was added after the basis was fixed):')
print(chk.to_string())

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/basis_shared.txt  (3,671 genes, the basis the model was fitted on)

recorded basis 3,671 | recomputed 3,668 | shared 3,577 (97.44% of the recorded)
forced panel genes: recorded 30, recomputed 30, identical: True
differences: 91 only recomputed, 94 only recorded; 119 of 185 are unannotated loci or antisense transcripts


fraction of cells detecting the gene -- differing genes: median 0.0007, max 0.0299; genes in both bases: median 0.0211
  most-detected differing genes: CFAP161 0.030, AC023424.3 0.019, LINC02224 0.015, AL049629.1 0.012, GCNA 0.011, AC011586.2 0.009
OK: the recorded basis is reproduced up to rarely-detected boundary genes

coverage of the current target panels by the basis (Fib_TGFb was added after the basis was fixed):
                    in_object  technical_excluded  target  in_basis                                           missing
panel                                                                                                                
id_perilobular             15                   0      15        15                                                  
id_interlobular            11                   0      11        11                                                  
id_medullary                8                   0       8         8                                      

## 4 · Write the factorisation inputs

The fit wants **raw counts** (pyliger normalises and scales per dataset itself); the gene basis is
applied inside the fit. The light companion object carries the cell metadata, the integrated
UMAP and the scVI latent space, and no expression matrix, so every later notebook loads it in
seconds. It also carries the current harmonised fibroblast annotation (`taa_l3_harmonized`, one
vocabulary in both arms) next to the original per-arm labels used during model selection.

In [11]:
harm = pd.read_csv(C.HARMONIZED_OBS, index_col=0, usecols=lambda c: c in ('', 'Unnamed: 0', C.HARMONIZED_LABEL))
adata.obs[C.HARMONIZED_LABEL] = harm[C.HARMONIZED_LABEL].reindex(adata.obs_names).astype(str).replace({'nan': np.nan}).values
print('harmonised label x arm:')
print(pd.crosstab(adata.obs[C.HARMONIZED_LABEL], adata.obs['arm']).to_string())

OBS_KEEP = ['arm', 'label', C.HARMONIZED_LABEL, 'donor', 'donor_comb', 'sample', 'study', 'batch', 'sort',
            'age', 'age_group', 'age_months', 'sex', 'chemistry_simple', 'n_counts', 'n_genes', 'adipose_mature']
OBS_KEEP = [c for c in OBS_KEEP if c in adata.obs.columns]

cin = ad.AnnData(X=adata.layers['counts'].copy(),
                 obs=adata.obs[['arm', 'label', 'donor_comb', 'age_group', 'study', 'batch']].copy(),
                 var=pd.DataFrame(index=adata.var_names))
cin.write_h5ad(f'{C.FINAL_DATA}/inmf_input_counts.h5ad', compression='gzip')
print(cin.shape, f'-> {C.FINAL_DATA}/inmf_input_counts.h5ad  ({os.path.getsize(C.FINAL_DATA + "/inmf_input_counts.h5ad") / 1e9:.2f} GB)')

light = ad.AnnData(X=sp.csr_matrix((adata.n_obs, 1), dtype=np.float32), obs=adata.obs[OBS_KEEP].copy())
for c in light.obs.columns:
    if light.obs[c].dtype == object:
        light.obs[c] = light.obs[c].astype(str).replace({'nan': np.nan})
light.obsm['X_umap'] = np.asarray(adata.obsm['X_umap'], dtype=np.float32)
light.obsm['X_scVI'] = np.asarray(adata.obsm['X_scVI'], dtype=np.float32)
light.write_h5ad(f'{C.FINAL_DATA}/fb_light.h5ad', compression='gzip')
print(light.shape, f'-> {C.FINAL_DATA}/fb_light.h5ad  ({os.path.getsize(C.FINAL_DATA + "/fb_light.h5ad") / 1e6:.0f} MB)')

harmonised label x arm:
arm                cells  nuclei
taa_l3_harmonized               
Fb-TGFB+            1284     273
Fb-adipo            5216     590
Fb-interlo         18363    4591
Fb-interm          14815    2926
Fb-med              7517    1260
Fb-perilo          23063    5218


(85116, 35530) -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/inmf_input_counts.h5ad  (0.40 GB)


(85116, 1) -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/fb_light.h5ad  (16 MB)


## Files written

| file | contents |
|---|---|
| `data/nmf_joint/final/basis_shared.txt` | the 3,671-gene basis the model was fitted on (copy of the recorded basis) |
| `data/nmf_joint/final/genes_basis_shared.csv` | the same, flagged shared-HVG vs panel-forced |
| `data/nmf_joint/final/basis_shared_recomputed.txt`, `genes_basis_shared_recomputed.csv` | the recomputation of this run (~97 % identical; differences are rarely detected boundary genes) |
| `data/nmf_joint/final/inmf_input_counts.h5ad` | raw counts, all genes, 85,116 cells — the fit input |
| `data/nmf_joint/final/fb_light.h5ad` | metadata + UMAP + scVI latent, no expression matrix |
| `results/final/01_gene_basis/*.csv` | detection rates, HVG ranks, depth sweep |
| `plots/final/01_gene_basis/*.pdf` | mature-adipose score by label and arm |